# Customer Churn Prediction (Python + Power BI)
Dataset: IBM Telco Customer Churn (7,043 customers)

## 1. Load and clean the data

In [ ]:
import pandas as pd

df = pd.read_csv("telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["Churn_Flag"] = (df["Churn"] == "Yes").astype(int)
df["TenureGroup"] = pd.cut(df["tenure"], bins=[-1,12,24,48,72], labels=["0-1 yr","1-2 yrs","2-4 yrs","4-6 yrs"])
df.head()

## 2. Quick exploration

In [ ]:
print("Churn rate:", round(df["Churn_Flag"].mean()*100,1), "%")
df.groupby("Contract")["Churn_Flag"].mean().round(3)

## 3. Train the model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = pd.get_dummies(df.drop(columns=["customerID","Churn","Churn_Flag","TenureGroup"]), drop_first=True)
y = df["Churn_Flag"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
print("Accuracy:", round(accuracy_score(y_test, model.predict(X_test)), 3))

## 4. Save predictions for Power BI

In [ ]:
df["Churn_Probability"] = model.predict_proba(X)[:, 1]
df["Risk_Level"] = pd.cut(df["Churn_Probability"], bins=[-0.01,0.3,0.6,1], labels=["Low","Medium","High"])
df.to_csv("churn_predictions.csv", index=False)
print("Saved churn_predictions.csv")